In [1]:
import pandas as pd
from shapely.geometry import Point
import geopandas as gpd
from pyproj import CRS
import hatyan
from opentnsim.graph.mixins import FIS
from opentnsim.graph.utils import find_closest_node, node_path_to_edge_path, get_sailing_distance
from opentnsim.graph.visualizations import plot_graph_folium
import numpy as np

In [2]:
df = pd.read_csv('C:\\users\\floorbakker\\documents\\RWS_data_water_levels_PoR.csv', delimiter=';', low_memory=False)

In [3]:
df['Timestamp'] = df['WAARNEMINGDATUM'] + ' ' + df['WAARNEMINGTIJD']
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d-%m-%Y %H:%M:%S')
df['Station'] = df['MEETPUNT_IDENTIFICATIE']
df['Value'] = df['NUMERIEKEWAARDE']/100
df['Geometry'] = gpd.points_from_xy(
    df['LON'],
    df['LAT']
)
crs_lookup = {
    epsg: CRS.from_user_input(epsg)
    for epsg in df['EPSG'].unique()
}

df['CRS'] = df['EPSG'].map(crs_lookup)

for crs, indices in df.groupby('CRS').groups.items():

    geometries = gpd.GeoSeries(
        df.loc[indices, 'Geometry'],
        crs=crs
    )

    transformed = geometries.to_crs("EPSG:4326")

    df.loc[indices, 'Geometry_WGS84'] = transformed.values

    print(f"\nSource CRS: {crs}")
    print(f"Target CRS: EPSG:4326")
    print(f"Identical: {geometries.equals(transformed)}")
    print(f"Source coordinates: {geometries.iloc[0].coords[0]}")
    print(f"Target coordinates: {transformed.iloc[0].coords[0]}")


Source CRS: ETRS89
Target CRS: EPSG:4326
Identical: True
Source coordinates: (4.312531, 51.891814)
Target coordinates: (4.312531, 51.891814)


In [4]:
meting_df = df[df.PROCES_TYPE == 'meting'].copy()
meting_df = meting_df[meting_df.Station !='Stellendam, buitenhaven']
water_level_df = meting_df.pivot(
    index='Timestamp',
    columns='Station',
    values='Value'
)

locations_dict = (
    meting_df.groupby('Station')['Geometry_WGS84']
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

water_level_df = water_level_df.dropna()

In [5]:
FG = FIS.load_network(version="0.3")
FG_copy = FG.copy()
FG_copy.remove_edge('B19664_A','B31010_B')
FG_copy.remove_edge('B42634_B','B42634_A')
FG_copy.remove_edge('B27780191_A','L4199_A')

In [6]:
node_dict = {}
for station, location in locations_dict.items():
    node_dict[station] = find_closest_node(FG, location)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [7]:
# manual corrections
node_dict['Hoek van Holland'] = '8862925'
node_dict['Parksluis'] = '8866388'
node_dict['Botlek Geulhaven'] = '8861158'
node_dict['Vlaardingen'] = '8864482'
node_dict['Maassluis'] = '8861674'
node_dict['Willemstad, Hollandsch Diep'] = '8860870'
node_dict['Moerdijk'] = '8860725'
node_dict['Dordrecht Oude Maas, Beneden Merwede'] = '30984396'

In [8]:
import logging
import pandas as pd
import hatyan

logging.getLogger('hatyan').setLevel(logging.WARNING)

component_df = pd.DataFrame()
meteorological_water_level_df = pd.DataFrame()
astronomic_water_level_df = pd.DataFrame()
for station in water_level_df.columns:
    ts = water_level_df[[station]].copy()
    ts.columns = ['values']
    
    ts.index = pd.DatetimeIndex(ts.index)
    ts = ts.sort_index()
    
    
    # Remove missing observations
    ts = ts.dropna()
    
    components = hatyan.analysis(
        ts,
        const_list='month'
    )
    times = ts.index
    components_attrs = components.attrs

    component_df[station + '_A'] = components['A']
    component_df[station + '_phi'] = components['phi_deg']

    # Construct Hatyan component DataFrame
    node = node_dict[station]
    comp = pd.DataFrame({
        'A': component_df[f'{station}_A'],
        'phi_deg': component_df[f'{station}_phi']
    }, index=component_df.index)
    comp.attrs = components_attrs
    
    prediction = hatyan.prediction(
        comp,
        times=times
    )

    meteorological_water_level_df[node] = water_level_df[[station]].rename(columns={station:'values'}) - prediction
    astronomic_water_level_df[node] = prediction
    
time_index = ts.index
components_attrs = components.attrs

In [9]:
import networkx as nx

measurement_nodes = list(set(node_dict.values()))

used_edges = set()

for i, source in enumerate(measurement_nodes):
    for target in measurement_nodes[i + 1:]:

        try:
            path = nx.shortest_path(
                FG_copy,
                source=source,
                target=target,
                weight='length_m'
            )

            used_edges.update(
                tuple(sorted(edge))
                for edge in zip(path[:-1], path[1:])
            )

        except nx.NetworkXNoPath:
            continue

# Create subgraph containing only selected route edges
FG_sub = FG_copy.edge_subgraph(used_edges).copy()

# Preserve measurement nodes, including any isolated nodes
FG_sub.add_nodes_from(measurement_nodes)

In [10]:
FG_sub.remove_edge('8866780','8864465')

In [11]:
# Create reverse mapping: station name -> node code
rename_dict = {}

for station, node_code in node_dict.items():
    rename_dict[f'{station}_A'] = f'{node_code}_A'
    rename_dict[f'{station}_phi'] = f'{node_code}_phi'

# Rename columns
component_df = component_df.rename(columns=rename_dict)

In [12]:
import networkx as nx
import numpy as np

def find_interpolation_weights(
    selected_nodes,
    FG_sub,
    power=3,
    k=4,
    max_distance=None
):

    interpolation_weights = {}

    # Calculate shortest network distances from every measurement node
    distances_from_sources = {}

    for source in selected_nodes:

        distances = nx.single_source_dijkstra_path_length(
            FG_sub,
            source,
            weight='length_m'
        )

        distances_from_sources[source] = distances

    # Calculate weights for every node without data
    for target in FG_sub.nodes:

        if target in selected_nodes:
            continue

        # Find distances to all reachable measurement nodes
        distances = {
            source: source_distances[target]
            for source, source_distances in distances_from_sources.items()
            if target in source_distances
        }

        if not distances:
            continue

        # Sort by distance
        distances = dict(
            sorted(distances.items(), key=lambda x: x[1])
        )

        # Keep only k nearest measurement nodes
        if k is not None:
            distances = dict(list(distances.items())[:k])

        # Apply maximum distance constraint
        if max_distance is not None:
            distances = {
                source: distance
                for source, distance in distances.items()
                if distance <= max_distance
            }

        if not distances:
            continue

        # Handle exact matches
        zero_distance_sources = [
            source for source, distance in distances.items()
            if distance == 0
        ]

        if zero_distance_sources:
            weights = {
                source: 1 / len(zero_distance_sources)
                for source in zero_distance_sources
            }

        else:
            # Inverse distance weighting
            raw_weights = {
                source: 1 / distance**power
                for source, distance in distances.items()
            }

            total_weight = sum(raw_weights.values())

            weights = {
                source: weight / total_weight
                for source, weight in raw_weights.items()
            }

        interpolation_weights[target] = {
            'sources': list(weights.keys()),
            'distances': list(distances.values()),
            'weights': list(weights.values())
        }

    return interpolation_weights

def interpolate_tidal_data_IDW(
    component_df,
    meteorological_water_level_df,
    astronomic_water_level_df,
    interpolation_weights,
    components_attrs
):

    new_components = {}
    new_meteo = {}
    new_astronomic = {}

    times = meteorological_water_level_df.index

    for node, info in interpolation_weights.items():

        sources = info['sources']
        weights = np.array(info['weights'])

        # Interpolate harmonic components
        Z_sources = []

        for source in sources:

            A = component_df[f'{source}_A']
            phi = component_df[f'{source}_phi']

            Z = A * np.exp(1j * np.deg2rad(phi))

            Z_sources.append(Z)

        Z_interpolated = sum(
            weight * Z
            for weight, Z in zip(weights, Z_sources)
        )

        new_components[f'{node}_A'] = np.abs(Z_interpolated)

        new_components[f'{node}_phi'] = np.mod(
            np.rad2deg(np.angle(Z_interpolated)),
            360
        )

        # Interpolate meteorological water level
        meteo_tide = sum(
            weight * meteorological_water_level_df[source]
            for weight, source in zip(weights, sources)
        )

        new_meteo[node] = meteo_tide

        # Calculate astronomical water level
        comp = pd.DataFrame({
            'A': new_components[f'{node}_A'],
            'phi_deg': new_components[f'{node}_phi']
        })

        comp.attrs = components_attrs.copy()

        prediction = hatyan.prediction(
            comp,
            times=times
        )

        if isinstance(prediction, pd.DataFrame):
            prediction = prediction['values']

        new_astronomic[node] = prediction

    # Concatenate all new columns at once
    if new_components:
        component_df = pd.concat(
            [
                component_df,
                pd.DataFrame(new_components, index=component_df.index)
            ],
            axis=1
        )

    if new_meteo:
        meteorological_water_level_df = pd.concat(
            [
                meteorological_water_level_df,
                pd.DataFrame(new_meteo, index=times)
            ],
            axis=1
        )

    if new_astronomic:
        astronomic_water_level_df = pd.concat(
            [
                astronomic_water_level_df,
                pd.DataFrame(new_astronomic, index=times)
            ],
            axis=1
        )

    return (
        component_df,
        meteorological_water_level_df,
        astronomic_water_level_df
    )

In [13]:
# Original measurement nodes
measurement_nodes = (
    component_df.columns.to_series()
    .str.replace(r'_(A|phi)$', '', regex=True)
    .drop_duplicates()
    .tolist()
)

# Calculate interpolation weights
interpolation_weights = find_interpolation_weights(
    selected_nodes=measurement_nodes,
    FG_sub=FG_sub,
    power=4,
    k=3,
    max_distance=50000
)

# Interpolate
(
    component_df,
    meteorological_water_level_df,
    astronomic_water_level_df
) = interpolate_tidal_data_IDW(
    component_df,
    meteorological_water_level_df,
    astronomic_water_level_df,
    interpolation_weights,
    components_attrs
)

In [14]:
def extrapolate_nearest_neighbour(
    FG,
    component_df,
    meteorological_water_level_df,
    astronomic_water_level_df,
    max_distance=10000
):

    # Nodes that already have data
    source_nodes = list(meteorological_water_level_df.columns)

    # Calculate shortest network distance to nearest source
    distances, paths = nx.multi_source_dijkstra(
        FG,
        sources=source_nodes,
        cutoff=max_distance,
        weight='length_m'
    )

    # Nodes that can be extrapolated
    new_nodes = set(distances) - set(source_nodes)

    new_components = {}
    new_meteo = {}
    new_astronomic = {}

    extrapolation_info = {}

    for target in new_nodes:

        # Nearest original source node
        source = paths[target][0]

        extrapolation_info[target] = {
            'source': source,
            'distance_m': distances[target]
        }

        # Copy harmonic components
        new_components[f'{target}_A'] = (
            component_df[f'{source}_A'].copy()
        )

        new_components[f'{target}_phi'] = (
            component_df[f'{source}_phi'].copy()
        )

        # Copy meteorological water level
        new_meteo[target] = (
            meteorological_water_level_df[source].copy()
        )

        # Copy astronomical water level
        new_astronomic[target] = (
            astronomic_water_level_df[source].copy()
        )

    # Append new data
    if new_components:
        component_df = pd.concat([
            component_df,
            pd.DataFrame(new_components, index=component_df.index)
        ], axis=1)

        meteorological_water_level_df = pd.concat([
            meteorological_water_level_df,
            pd.DataFrame(new_meteo, index=meteorological_water_level_df.index)
        ], axis=1)

        astronomic_water_level_df = pd.concat([
            astronomic_water_level_df,
            pd.DataFrame(new_astronomic, index=astronomic_water_level_df.index)
        ], axis=1)

    print(f"Extrapolated nodes: {len(new_nodes)}")
    print(f"Nodes without data: {FG.number_of_nodes() - len(source_nodes) - len(new_nodes)}")

    return (
        component_df,
        meteorological_water_level_df,
        astronomic_water_level_df,
        extrapolation_info
    )


In [15]:
FG_clean = FG.copy()

nodes_to_remove = [
    node for node in FG_clean.nodes
    if ('L' in str(node) or 'S' in str(node)) and node not in FG_sub.nodes
]

FG_clean.remove_nodes_from(nodes_to_remove)

In [16]:
(
    component_df,
    meteorological_water_level_df,
    astronomic_water_level_df,
    extrapolation_info
) = extrapolate_nearest_neighbour(
    FG=FG_clean,
    component_df=component_df,
    meteorological_water_level_df=meteorological_water_level_df,
    astronomic_water_level_df=astronomic_water_level_df,
    max_distance=10000
)

Extrapolated nodes: 400
Nodes without data: 9614


In [17]:
actual_water_level_df = astronomic_water_level_df + meteorological_water_level_df

In [18]:
# Nodes for which we have data
nodes_with_data = set(
    meteorological_water_level_df.columns
)

# Create subgraph from original graph
FG_data = FG.subgraph(nodes_with_data).copy()

In [19]:
node_coordinates = {
    node: (data['X'], data['Y'])
    for node, data in FG_data.nodes(data=True)
}

In [29]:
component_df

,8861158_A,8861158_phi,22638228_A,22638228_phi,30984396_A,30984396_phi,B15339_B_A,B15339_B_phi,B23888_A_A,B23888_A_phi,...,8865802_A,8865802_phi,8860787_A,8860787_phi,30984595_A,30984595_phi,8861807_A,8861807_phi,8864041_A,8864041_phi
A0,0.224288,0.000000,0.767021,0.000000,0.459865,0.000000,0.189589,0.000000,0.245796,0.000000,...,0.233093,0.000000,0.235193,0.000000,0.526341,0.000000,0.522959,0.000000,0.765266,0.000000
Q1,0.045890,138.478984,0.017576,239.817052,0.019313,179.297823,0.045444,137.256372,0.040675,139.839534,...,0.044117,140.471873,0.044039,140.842640,0.014741,232.576507,0.014889,230.788564,0.017535,239.560857
O1,0.091746,201.687450,0.016324,252.895552,0.036773,228.300536,0.087465,200.272405,0.080201,202.981718,...,0.089274,203.627935,0.089218,204.029787,0.018556,288.030352,0.020675,292.695307,0.016399,252.713496
K1,0.059719,43.216499,0.009161,121.909141,0.021684,69.620991,0.056928,42.835459,0.051655,44.363965,...,0.057688,44.739307,0.057916,45.230850,0.012950,125.214177,0.014654,115.701439,0.009186,121.435788
3MS2,0.022754,303.499384,0.008601,46.405750,0.009851,344.506041,0.021599,295.312076,0.019591,307.320549,...,0.023636,308.051035,0.024138,309.753622,0.009526,66.507682,0.010318,70.983637,0.008584,46.179261
MNS2,0.014908,160.612261,0.008317,282.284374,0.009136,233.261411,0.014205,157.740305,0.013831,168.465703,...,0.016486,167.861778,0.016697,168.883162,0.009133,292.109424,0.008817,286.081284,0.008308,282.087455
MU2,0.088574,189.272827,0.022443,285.626004,0.040754,223.398574,0.088571,180.555107,0.079072,189.186576,...,0.088555,195.147788,0.089615,196.582661,0.012825,295.415938,0.014636,317.624539,0.022413,285.220648
N2,0.111656,49.748914,0.022446,146.809945,0.048847,84.028978,0.110032,44.679916,0.104481,49.711281,...,0.111478,54.911008,0.111724,55.969659,0.012301,168.797626,0.019721,169.548918,0.022430,146.329878
M2,0.802547,102.804241,0.147358,200.713574,0.332456,139.199079,0.800083,97.970715,0.731514,104.218551,...,0.799453,108.161047,0.805134,109.385653,0.116037,213.563820,0.140745,220.338385,0.147360,200.218886
2MN2,0.093891,336.324581,0.020711,81.022202,0.043233,21.951332,0.093992,332.702162,0.084437,339.256332,...,0.094480,342.802815,0.095484,344.161231,0.020181,70.713530,0.016005,74.020757,0.020717,80.545968


In [20]:
import folium
import branca
import numpy as np
import pandas as pd
import json
import base64

from branca.element import Element

def animate_tidal_data(df, background_map = 'default'):

    # ==========================================================
    # 1. PREPARE DATA
    # ==========================================================
    
    df = df.sort_index().copy()
    
    # Keep only nodes with available coordinates
    nodes = [
        node for node in df.columns
        if node in node_coordinates
    ]
    
    # Convert water levels to float32
    # This substantially reduces storage requirements.
    values = (
        df[nodes]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype="<f4")
    )
    
    n_times, n_nodes = values.shape
    
    if n_times == 0:
        raise ValueError("water_level_df contains no timestamps.")
    if n_nodes == 0:
        raise ValueError(
            "No water-level columns have matching entries in node_coordinates."
        )
    
    timestamps = [
        pd.Timestamp(t).strftime("%Y-%m-%d %H:%M")
        for t in df.index
    ]
    
    # Network coordinates
    coords = [
        [float(node_coordinates[node][0]),
         float(node_coordinates[node][1])]
        for node in nodes
    ]
    
    # Map node names to numerical indices
    node_index = {
        node: i for i, node in enumerate(nodes)
    }
    
    # ==========================================================
    # 2. PREPARE EDGES USING ACTUAL LINESTRING GEOMETRY
    # ==========================================================
    
    edges = []
    
    for n1, n2, attrs in FG_data.edges(data=True):
    
        if n1 not in node_index or n2 not in node_index:
            continue
    
        geometry = attrs.get("geometry")
    
        if geometry is None:
            continue
    
        # Extract ALL vertices from the original LineString
        edge_coords = [
            [float(x), float(y)]
            for x, y in geometry.coords
        ]
    
        edges.append({
            "start": node_index[n1],
            "end": node_index[n2],
            "coordinates": edge_coords
        })
    
    n_edges = len(edges)
    
    print(f"Prepared {n_edges} edges")
    
    # ==========================================================
    # 3. ENCODE WATER LEVELS AS BINARY DATA
    # ==========================================================
    
    # Flatten the matrix in time-major order.
    # NaN values remain NaN in the Float32Array.
    
    binary_data = values.tobytes(order="C")
    
    encoded_data = base64.b64encode(
        binary_data
    ).decode("ascii")
    
    # ==========================================================
    # 4. COLOUR SCALE
    # ==========================================================
    
    valid_values = values[np.isfinite(values)]
    
    if len(valid_values) == 0:
        raise ValueError("No valid water level values found.")
    
    vmin = float(valid_values.min())
    vmax = float(valid_values.max())
    
    colors = [
        "#b2182b",  # Dark red
        "#ef8a62",  # Light red
        "#f7f7f7",  # White
        "#67a9cf",  # Light blue
        "#2166ac"   # Dark blue
    ]
    
    colormap = branca.colormap.LinearColormap(
        colors=colors,
        vmin=vmin,
        vmax=vmax,
        caption="Water level (m)"
    )
    
    # ==========================================================
    # 5. CREATE FOLIUM MAP
    # ==========================================================
    
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    
    center = [
        float(np.mean(lats)),
        float(np.mean(lons))
    ]
    
    m = folium.Map(
        location=[52.0, 4.3],
        zoom_start=10,
        tiles=None
    )

    if background_map == 'grey':
        folium.TileLayer(
            tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
            attr='Tiles &copy; Esri',
            name='Esri Light Gray',
            overlay=False,
            control=True
        ).add_to(m)

    else:
        folium.TileLayer(
            tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
            attr='Tiles &copy; Esri',
            name='Satellite imagery',
            overlay=False,
            control=True
        ).add_to(m)
        
    # Fit map to the network
    m.fit_bounds([
        [min(lats), min(lons)],
        [max(lats), max(lons)]
    ])
    
    colormap.add_to(m)
    
    # ==========================================================
    # 6. ADD TIME SLIDER HTML
    # ==========================================================
    
    control_html = """
    <div id="time-control"
         style="
            position: fixed;
            bottom: 25px;
            left: 50%;
            transform: translateX(-50%);
            z-index: 9999;
            background: white;
            padding: 12px 18px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.3);
            width: 65%;
            min-width: 320px;
            max-width: 850px;
            font-family: Arial;
         ">
    
        <div style="
            display:flex;
            justify-content:space-between;
            align-items:center;
            margin-bottom:8px;
        ">
    
            <button id="play-button">
                Play
            </button>
    
            <strong id="current-time">
                Loading...
            </strong>
    
            <select id="speed">
                <option value="1000">1x</option>
                <option value="500">2x</option>
                <option value="250" selected>4x</option>
                <option value="100">10x</option>
                <option value="50">20x</option>
                <option value="20">50x</option>
            </select>
    
        </div>
    
        <input
            type="range"
            id="time-slider"
            min="0"
            max="0"
            value="0"
            step="1"
            style="width:100%;"
        >
    
        <div style="
            display:flex;
            justify-content:space-between;
            font-size:11px;
            color:#666;
        ">
            <span id="start-time"></span>
            <span id="frame-counter"></span>
            <span id="end-time"></span>
        </div>
    
    </div>
    """
    
    m.get_root().html.add_child(
        Element(control_html)
    )
    
    # Keep the large Base64 payload out of executable JavaScript.
    # This reduces JavaScript parsing time in the browser.
    m.get_root().html.add_child(
        Element(
            '<script id="water-level-data" type="application/octet-stream">'
            + encoded_data +
            '</script>'
        )
    )
    
    # ==========================================================
    # 7. JAVASCRIPT ANIMATION
    # ==========================================================
    
    js = r"""
    (function() {
      const mapName = __MAP_NAME__;
    
      function waitForMap(attempt) {
        const map = window[mapName];
    
        if (!map) {
          if (attempt < 100) {
            window.setTimeout(function() {
              waitForMap(attempt + 1);
            }, 100);
          } else {
            const status = document.getElementById("current-time");
            if (status) {
              status.textContent =
                "Animation error: Folium map '" + mapName +
                "' was not found after 10 seconds. Check that the map script loaded.";
              status.style.color = "#b00020";
            }
            console.error("Folium map was not found on window:", mapName);
          }
          return;
        }
    
        try {
          const statusElement = document.getElementById("current-time");
          if (statusElement) statusElement.textContent = "Preparing animation data...";
    
        const timestamps = __TIMESTAMPS__;
    
        const coordinates = __COORDS__;
    
        const nodes = __NODES__;
    
        const edges = __EDGES__;
    
        const nTimes = __NTIMES__;
    
        const nNodes = __NNODES__;
    
        const vmin = __VMIN__;
    
        const vmax = __VMAX__;
    
        const palette = [
            [178, 24, 43],    // Dark red
            [239, 138, 98],   // Light red
            [247, 247, 247],  // White
            [103, 169, 207],  // Light blue
            [33, 102, 172]    // Dark blue
        ];
    
        // ------------------------------------------------------
        // Decode binary water-level data
        // ------------------------------------------------------
    
        const dataElement = document.getElementById("water-level-data");
        if (!dataElement) {
            throw new Error("Water-level data block was not found in the HTML.");
        }
    
        const encoded = dataElement.textContent.trim();
        if (!encoded) {
            throw new Error("Water-level data block is empty.");
        }
    
        const raw = atob(encoded);
        const bytes = Uint8Array.from(raw, character => character.charCodeAt(0));
    
        if (bytes.byteLength % Float32Array.BYTES_PER_ELEMENT !== 0) {
            throw new Error("Decoded data size is not a multiple of 4 bytes.");
        }
    
        const data = new Float32Array(
            bytes.buffer,
            bytes.byteOffset,
            bytes.byteLength / Float32Array.BYTES_PER_ELEMENT
        );
    
        const expectedLength = nTimes * nNodes;
        if (data.length !== expectedLength) {
            throw new Error(
                "Data size mismatch: expected " + expectedLength +
                " values, but decoded " + data.length + "."
            );
        }
    
        if (!nTimes || !nNodes) {
            throw new Error("No timestamps or mapped nodes are available.");
        }
    
        // ------------------------------------------------------
        // Colour interpolation
        // ------------------------------------------------------
    
        function getColor(value) {
    
            let t = (vmax === vmin)
                ? 0.5
                : (value - vmin) / (vmax - vmin);
    
            t = Math.max(0, Math.min(1, t));
    
            const segment = t * 4;
    
            const index = Math.min(3, Math.floor(segment));
    
            const fraction = segment - index;
    
            const c1 = palette[index];
            const c2 = palette[index + 1];
    
            const r = Math.round(
                c1[0] + fraction * (c2[0] - c1[0])
            );
    
            const g = Math.round(
                c1[1] + fraction * (c2[1] - c1[1])
            );
    
            const b = Math.round(
                c1[2] + fraction * (c2[2] - c1[2])
            );
    
            return `rgb(${r},${g},${b})`;
        }
    
        // ------------------------------------------------------
        // Escape popup content
        // ------------------------------------------------------
    
        function escapeHTML(value) {
            return String(value).replace(/[&<>"']/g, function(c) {
                return {
                    '&': '&amp;',
                    '<': '&lt;',
                    '>': '&gt;',
                    '"': '&quot;',
                    "'": '&#39;'
                }[c];
            });
        }
    
        // ------------------------------------------------------
        // Create node layer ONCE
        // ------------------------------------------------------
    
        const nodeGroup = L.layerGroup().addTo(map);
    
        const nodeLayers = [];
    
        nodes.forEach((node, i) => {
    
            const lon = coordinates[i][0];
            const lat = coordinates[i][1];
    
            const marker = L.circleMarker(
                [lat, lon],
                {
                    radius: 5,
                    color: "black",
                    weight: 1,
                    fillColor: "blue",
                    fillOpacity: 0.9
                }
            );
    
            marker.bindPopup("");
    
            marker.addTo(nodeGroup);
    
            nodeLayers.push(marker);
        });
    
        // ------------------------------------------------------
        // Create edge layer ONCE
        // ------------------------------------------------------
        
        const edgeGroup = L.layerGroup().addTo(map);
        
        const edgeLayers = [];
    
        edges.forEach(edge => {
        
            // Convert [longitude, latitude] to [latitude, longitude]
            const latlon = edge.coordinates.map(
                p => [p[1], p[0]]
            );
        
            const line = L.polyline(latlon, {
                color: "blue",
                weight: 3,
                opacity: 0.85
            });
        
            line.bindPopup("");
            line.addTo(edgeGroup);
        
            edgeLayers.push(line);
        });
    
        // ------------------------------------------------------
        // Layer control
        // ------------------------------------------------------
    
        L.control.layers(
            null,
            {
                "Nodes": nodeGroup,
                "Edges": edgeGroup
            },
            {
                collapsed: true
            }
        ).addTo(map);
    
        // ------------------------------------------------------
        // Update network for selected timestamp
        // ------------------------------------------------------
    
        function updateFrame(t) {
    
            const offset = t * nNodes;
    
            // Update nodes
            for (let i = 0; i < nNodes; i++) {
    
                const value = data[offset + i];
    
                const marker = nodeLayers[i];
    
                if (!Number.isFinite(value)) {
    
                    marker.setStyle({
                        opacity: 0,
                        fillOpacity: 0
                    });
    
                    continue;
                }
    
                const color = getColor(value);
    
                marker.setStyle({
                    color: "black",
                    fillColor: color,
                    opacity: 1,
                    fillOpacity: 0.9
                });
    
                marker.setPopupContent(
                    "<b>Node:</b> " +
                    escapeHTML(nodes[i]) +
                    "<br><b>Water level:</b> " +
                    value.toFixed(3) +
                    " m"
                );
            }
    
            // Update edges using endpoint averages
            for (let k = 0; k < edges.length; k++) {
    
                const i = edges[k].start;
                const j = edges[k].end;
    
                const h1 = data[offset + i];
                const h2 = data[offset + j];
    
                const line = edgeLayers[k];
    
                if (!Number.isFinite(h1) ||
                    !Number.isFinite(h2)) {
    
                    line.setStyle({
                        opacity: 0
                    });
    
                    continue;
                }
    
                const edgeValue = (h1 + h2) / 2;
    
                const color = getColor(edgeValue);
    
                line.setStyle({
                    color: color,
                    weight: 4,
                    opacity: 0.85
                });
    
                line.setPopupContent(
                    "<b>" +
                    escapeHTML(nodes[i]) +
                    " - " +
                    escapeHTML(nodes[j]) +
                    "</b><br>" +
                    "<b>Average water level:</b> " +
                    edgeValue.toFixed(3) +
                    " m"
                );
            }
    
            // Update time display
            document.getElementById("current-time").textContent =
                timestamps[t];
    
            document.getElementById("frame-counter").textContent =
                (t + 1) + " / " + nTimes;
    
            document.getElementById("time-slider").value = t;
        }
    
        // ------------------------------------------------------
        // Slider
        // ------------------------------------------------------
    
        const slider = document.getElementById("time-slider");
    
        slider.max = nTimes - 1;
    
        document.getElementById("start-time").textContent =
            timestamps[0];
    
        document.getElementById("end-time").textContent =
            timestamps[nTimes - 1];
    
        slider.addEventListener("input", function() {
    
            pauseAnimation();
    
            updateFrame(Number(this.value));
    
        });
    
        // ------------------------------------------------------
        // Play / pause
        // ------------------------------------------------------
    
        let animationTimer = null;
    
        const playButton = document.getElementById("play-button");
    
        function pauseAnimation() {
    
            if (animationTimer !== null) {
    
                clearInterval(animationTimer);
    
                animationTimer = null;
            }
    
            playButton.textContent = "Play";
        }
    
        playButton.addEventListener("click", function() {
    
            if (animationTimer !== null) {
    
                pauseAnimation();
    
                return;
            }
    
            if (Number(slider.value) >= nTimes - 1) {
    
                updateFrame(0);
            }
    
            playButton.textContent = "Pause";
    
            const interval = Number(
                document.getElementById("speed").value
            );
    
            animationTimer = setInterval(function() {
    
                let next = Number(slider.value) + 1;
    
                if (next >= nTimes) {
    
                    pauseAnimation();
    
                    return;
                }
    
                updateFrame(next);
    
            }, interval);
    
        });
    
        // ------------------------------------------------------
        // Initialize
        // ------------------------------------------------------
    
        updateFrame(0);
    
        // Remove the large encoded text from the DOM after successful decoding.
        dataElement.textContent = "";
    
        } catch (error) {
          console.error("Water-level animation initialization failed:", error);
          const status = document.getElementById("current-time");
          if (status) {
            status.textContent = "Animation error: " + (error.message || error);
            status.style.color = "#b00020";
          }
        }
      }
    
      // Folium's map variable is created by its own script. Wait for it to exist
      // before creating the animation layers.
      waitForMap(0);
    })();
    """
    
    # ==========================================================
    # 8. INSERT DATA INTO JAVASCRIPT
    # ==========================================================
    
    replacements = {
        "__MAP_NAME__": json.dumps(m.get_name()),
        "__TIMESTAMPS__": json.dumps(timestamps),
        "__COORDS__": json.dumps(coords, separators=(",", ":")),
        "__NODES__": json.dumps(
            [str(n) for n in nodes],
            separators=(",", ":")
        ),
        "__EDGES__": json.dumps(
            edges,
            separators=(",", ":")
        ),
        "__NTIMES__": str(n_times),
        "__NNODES__": str(n_nodes),
        "__VMIN__": repr(vmin),
        "__VMAX__": repr(vmax)
    }
    
    for key, value in replacements.items():
        js = js.replace(key, value)
    
    # Add JavaScript after Folium initializes the map
    m.get_root().script.add_child(
        Element(js)
    )
    
    # Display map in Jupyter
    return m

In [27]:
data = 'actual water level'

if data == 'actual water level':
    df = actual_water_level_df
if data == 'meteorological water level':
    df = meteorological_water_level_df
if data == 'astronomical water level':
    df = astronomic_water_level_df

m = animate_tidal_data(df)
m

Prepared 673 edges


In [24]:
m.save("tidal_network_map_PoR.html")

In [41]:
actual_water_level_df.to_pickle('actual_water_levels_network_PoR.pkl')